# Rule-Based Evaluation Pipeline

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('Complaints.csv')
df = df[df['complaint_date'] != 'complaint_date'].copy()

orders_df = pd.read_csv('Orders.csv')
df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
df = df.sort_values(by=['customer_id', 'complaint_date']).reset_index(drop=True)
df['prior_complaints'] = df.groupby('customer_id').cumcount()
order_counts = orders_df.groupby('customer_id').size().to_dict()
df['total_orders'] = df['customer_id'].map(order_counts).fillna(0)
df['complaint_ratio'] = (df['prior_complaints'] + 1) / df['total_orders'].apply(lambda x: 1 if x == 0 else x)
test_df = df.sample(frac=0.2, random_state=42).copy()

In [2]:
test_unlabeled = test_df[['complaint_id', 'complaint_text', 'customer_id', 'prior_complaints', 'complaint_ratio', 'total_orders']].copy()
test_unlabeled.head()

,complaint_id,complaint_text,customer_id,prior_complaints,complaint_ratio,total_orders
361,CMP-493,screen showing lines,C-071,3,0.500,8
73,CMP-302,screen ghost touch happening randomly,C-014,4,0.625,8
374,CMP-262,screen touch not working in some areas,C-074,1,0.250,8
155,CMP-455,audio very low even at max,C-029,6,0.875,8
104,CMP-471,device rebooting randomly,C-019,6,0.875,8


In [3]:
def compute_fraud_features(row):
    text = str(row['complaint_text']).lower()
    score = 6
    classification = "Legitimate"
    
    if 'box' in text and any(w in text for w in ['random', 'broken', 'scratches', 'tampered', 'older', 'different']):
        score, classification = 95, 'Fraud'
    elif 'ordered' in text and ('got' in text or 'looks' in text or 'why' in text):
        score, classification = 93, 'Fraud'
    elif 'used' in text and ('already' in text or 'returning' in text or 'few' in text or 'event' in text or 'now' in text):
        score, classification = 85, 'Fraud'
    elif any(phrase in text for phrase in ['wrong item', 'wrong model', 'different from', 'not matching', 'match description']): 
        score, classification = 90, 'Fraud'
    elif any(phrase in text for phrase in ['changed my mind', 'dont like', "don't like", 'prefer', 'not my style', 'expected better', 'not what i expected', 'feels cheap', 'not worth']):
        score, classification = 18, 'Fraud'
    else:
        words = text.split()
        if text.strip() in ['ok', 'bad product', 'not good', 'worst', 'bad', 'meh'] or (len(words) <= 3 and 'dead' not in text and 'not working' not in text):
            score, classification = 22, 'Fraud'


    prior_complaints = row['prior_complaints']
    ratio = row['complaint_ratio']
    
    if prior_complaints >= 3:
        score += 15
        classification = "Fraud"
        
    if ratio > 0.5 and row['total_orders'] > 2:
        score += 10
        classification = "Fraud"

    if ratio <= 0.1 and prior_complaints == 0:
        score -= 5  
        
   
    score = max(0, min(100, score))
    if score >= 80:
        classification = "Fraud"
        
    return classification


test_unlabeled['Predicted_Fraud_Class'] = test_unlabeled.apply(compute_fraud_features, axis=1)
test_unlabeled.head()

,complaint_id,complaint_text,customer_id,prior_complaints,complaint_ratio,total_orders,Predicted_Fraud_Class
361,CMP-493,screen showing lines,C-071,3,0.500,8,Fraud
73,CMP-302,screen ghost touch happening randomly,C-014,4,0.625,8,Fraud
374,CMP-262,screen touch not working in some areas,C-074,1,0.250,8,Legitimate
155,CMP-455,audio very low even at max,C-029,6,0.875,8,Fraud
104,CMP-471,device rebooting randomly,C-019,6,0.875,8,Fraud


In [4]:

accuracy = accuracy_score(test_df['fraud_classification'], test_unlabeled['Predicted_Fraud_Class'])

print(f" Accuracy: {accuracy * 100:.2f}%")
print("\nOverall Report:\n")
print(classification_report(test_df['fraud_classification'], test_unlabeled['Predicted_Fraud_Class']))

 Accuracy: 59.00%

Overall Report:

              precision    recall  f1-score   support

       Fraud       0.50      0.78      0.61        41
  Legitimate       0.75      0.46      0.57        59

    accuracy                           0.59       100
   macro avg       0.62      0.62      0.59       100
weighted avg       0.65      0.59      0.59       100

